# 01 - Limpeza e Tratamento dos Dados

Responsavel: Integrante 1

Objetivo: carregar o dataset bruto, diagnosticar problemas de qualidade e aplicar os tratamentos descritos em `PLANO_DE_TRATAMENTO.md`. Cada tratamento deve vir acompanhado de uma celula de texto com a JUSTIFICATIVA e o IMPACTO esperado.

Saida: `data/dataset_tratado.csv`.

In [20]:
import sys
print(sys.executable)

/usr/bin/python3


In [21]:
import os, subprocess

REPO_PATH = '/content/super-projeto-de-probabilidade-estatistica'

if os.path.exists(REPO_PATH):
    os.chdir(REPO_PATH)
    subprocess.run(['git', 'pull', 'origin', 'master'], check=True)

print('Diretório atual:', os.getcwd())

Diretório atual: /content/super-projeto-de-probabilidade-estatistica


In [22]:
# Setup
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

CAMINHO = 'CSV/ncr_ride_bookings.csv'
df = pd.read_csv(CAMINHO)
df.shape

(150000, 21)

## 1. Diagnostico inicial

In [23]:
# Visao geral
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,NaN,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,NaN,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,NaN,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [24]:
# Tipos, nulos e cardinalidade
df.info()
# df.isna().sum()
# df.nunique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Date                               150000 non-null  object 
 1   Time                               150000 non-null  object 
 2   Booking ID                         150000 non-null  object 
 3   Booking Status                     150000 non-null  object 
 4   Customer ID                        150000 non-null  object 
 5   Vehicle Type                       150000 non-null  object 
 6   Pickup Location                    150000 non-null  object 
 7   Drop Location                      150000 non-null  object 
 8   Avg VTAT                           139500 non-null  float64
 9   Avg CTAT                           102000 non-null  float64
 10  Cancelled Rides by Customer        10500 non-null   float64
 11  Reason for cancelling by Customer  1050

## 2. Variável alvo

Derivar a alvo binária a partir de `Booking Status`: **Concluída (1)** vs **Não Concluída (0)**.

**Justificativa:** O dataset tem 5 categorias (*Completed*, *Cancelled by Driver*, *No Driver Found*, *Cancelled by Customer*, *Incomplete*). Agrupar todos os desfechos negativos em uma única classe simplifica a modelagem e mantém a interpretabilidade do Teorema de Bayes.

**Impacto esperado:** ~62% Concluídas vs ~38% Não Concluídas — desbalanceamento moderado, aceitável sem over/undersampling.

In [ ]:
print('Categorias de Booking Status:')
print(df['Booking Status'].value_counts(dropna=False))
print()

# Completed -> 1 (Concluida); todos os outros desfechos -> 0 (Nao concluida)
df['alvo'] = (df['Booking Status'] == 'Completed').astype(int)

total = len(df)
print(f'Concluidas     (1): {df["alvo"].sum():>7,}  ({df["alvo"].mean():.1%})')
print(f'Nao concluidas (0): {(df["alvo"]==0).sum():>7,}  ({1-df["alvo"].mean():.1%})')

## 3. Tratamentos

Para CADA tratamento abaixo, escreva uma celula de texto com: problema -> acao -> justificativa -> impacto. Veja a tabela no PLANO_DE_TRATAMENTO.md.

### 3.1 Remover vazamento de dados (data leakage) e colunas irrelevantes

**Problema:** Várias colunas só existem *após* o desfecho da corrida — usá-las no modelo seria trapacear: o modelo aprenderia o resultado, não o predito.
- `Avg CTAT`, `Booking Value`, `Ride Distance`, `Payment Method`, `Driver Ratings`, `Customer Rating` — só preenchidos em corridas Concluídas.
- Colunas de cancelamento/incompleto — só preenchidas quando o desfecho já aconteceu.
- `Booking ID` e `Customer ID` — identificadores únicos sem valor preditivo (cardinalidade >148k).

**Ação:** Remover todas essas colunas; `Booking Status` também é removida após criar `alvo`.

**Impacto:** Dataset reduz de 21 para 6 colunas — apenas features disponíveis *antes* do desfecho.

In [ ]:
colunas_remover = [
    'Booking Status',                       # fonte da variavel alvo — removida apos uso
    'Avg CTAT',                             # so preenchida em corridas concluidas (leakage)
    'Booking Value',                        # valor definido apenas ao concluir
    'Ride Distance',                        # distancia real so disponivel ao final
    'Payment Method',                       # pagamento so ocorre apos conclusao
    'Driver Ratings',                       # avaliacao pos-corrida
    'Customer Rating',                      # avaliacao pos-corrida
    'Cancelled Rides by Customer',
    'Reason for cancelling by Customer',
    'Cancelled Rides by Driver',
    'Driver Cancellation Reason',
    'Incomplete Rides',
    'Incomplete Rides Reason',
    'Booking ID',                           # identificador unico — sem valor preditivo
    'Customer ID',                          # cardinalidade >148k valores unicos
]

df = df.drop(columns=[c for c in colunas_remover if c in df.columns])
print('Colunas restantes:', df.columns.tolist())
print('Shape:', df.shape)

### 3.2 Valores ausentes

**Problema:** `Avg VTAT` tem 10.500 nulos — exatamente as corridas *No Driver Found* (nenhum motorista apareceu, logo não há tempo de chegada registrado). Nulo aqui é estrutural, não aleatório.

**Ação:** Criar flag binária `sem_vtat` (1 = sem motorista) para preservar essa informação preditiva; depois preencher os nulos com a mediana da coluna.

**Justificativa:** Mediana é robusta a assimetrias; o flag `sem_vtat` garante que o modelo aprenda que "sem VTAT" é por si só um sinal forte de não-conclusão.

**Impacto:** Zero nulos restantes; feature `sem_vtat` adicionada.

In [ ]:
print('Nulos antes do tratamento:')
print(df.isna().sum())
print()

# Avg VTAT: nulos estruturais (corridas sem motorista)
df['sem_vtat'] = df['Avg VTAT'].isna().astype(int)

mediana_vtat = df['Avg VTAT'].median()
df['Avg VTAT'] = df['Avg VTAT'].fillna(mediana_vtat)

print(f'Avg VTAT: {df["sem_vtat"].sum()} nulos preenchidos com mediana = {mediana_vtat:.1f} min')
print(f'Flag sem_vtat: {df["sem_vtat"].sum()} corridas sem motorista marcadas')
print()
print('Nulos apos tratamento:')
print(df.isna().sum())

### 3.3 Tipos e datas — engenharia de atributos temporais

**Problema:** `Date` e `Time` são strings brutas sem valor preditivo direto; o padrão de corridas varia fortemente por hora do dia, dia da semana e mês.

**Ação:** Converter para datetime e extrair `hora`, `dia_semana`, `mes` e `periodo_dia` (manha/tarde/noite/madrugada). Remover as colunas originais.

**Justificativa:** Features temporais numéricas são interpretáveis pelos modelos; `periodo_dia` é especialmente útil para as tabelas de contingência do Bayes manual.

**Impacto:** 4 features novas; `Date` e `Time` removidas.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S', errors='coerce')

df['hora']       = df['Time'].dt.hour
df['dia_semana'] = df['Date'].dt.dayofweek   # 0 = segunda, 6 = domingo
df['mes']        = df['Date'].dt.month

def periodo_dia(h):
    if pd.isna(h):    return 'desconhecido'
    if 5  <= h < 12:  return 'manha'
    if 12 <= h < 18:  return 'tarde'
    if 18 <= h < 23:  return 'noite'
    return 'madrugada'

df['periodo_dia'] = df['hora'].apply(periodo_dia)
df = df.drop(columns=['Date', 'Time'])

print('Features temporais criadas:')
print(df[['hora', 'dia_semana', 'mes', 'periodo_dia']].head(3))
print()
print('Distribuicao de periodo_dia:')
print(df['periodo_dia'].value_counts())

### 3.4 Duplicatas

**Problema:** O dataset original tem 150.000 linhas mas apenas 148.767 `Booking ID` únicos — existem registros duplicados.

**Ação:** `drop_duplicates()` sobre todas as colunas restantes (Booking ID já foi removido na etapa 3.1).

**Justificativa:** Duplicatas inflam artificialmente o treino e distorcem métricas de avaliação.

**Impacto:** Remoção de ~1.200 linhas redundantes.

In [ ]:
n_antes = len(df)
df = df.drop_duplicates()
n_removidas = n_antes - len(df)
print(f'Linhas antes:          {n_antes:,}')
print(f'Duplicatas removidas:  {n_removidas:,}')
print(f'Linhas apos:           {len(df):,}')

### 3.5 Outliers

**Problema:** `Avg VTAT` poderia conter tempos de espera implausíveis (negativos ou excessivos).

**Ação:** Calcular limites por IQR (fator 1.5) e aplicar *clipping* — valores fora do intervalo são substituídos pelo limite, sem remover linhas.

**Justificativa:** Clipping preserva todas as observações; remoção de linhas seria excessiva aqui pois o range original (2–20 min) já é razoável.

**Impacto:** Esperado zero ou poucos valores alterados.

In [ ]:
Q1 = df['Avg VTAT'].quantile(0.25)
Q3 = df['Avg VTAT'].quantile(0.75)
IQR = Q3 - Q1
lower = max(0.0, Q1 - 1.5 * IQR)
upper = Q3 + 1.5 * IQR

n_out = ((df['Avg VTAT'] < lower) | (df['Avg VTAT'] > upper)).sum()
print(f'Avg VTAT — Q1={Q1:.1f}  Q3={Q3:.1f}  IQR={IQR:.1f}')
print(f'Limites IQR: [{lower:.1f}, {upper:.1f}]')
print(f'Valores fora do intervalo: {n_out}')

df['Avg VTAT'] = df['Avg VTAT'].clip(lower=lower, upper=upper)
print('Clipping aplicado.')

### 3.6 Padronização de categorias

**Problema:** `Booking ID` e `Customer ID` foram lidos com aspas duplas no CSV (`"CNR5884300"`). Colunas texto podem ter espaços extras que criam categorias duplicadas invisíveis.

**Ação:** Strip de espaços e aspas em todas as colunas de texto.

**Justificativa:** Aspas e espaços extras causariam agrupamentos incorretos nos notebooks de Bayes e EDA.

**Impacto:** Categorias limpas e consistentes em `Vehicle Type`, `Pickup Location`, `Drop Location`.

In [ ]:
for col in df.select_dtypes(include=['object', 'str']).columns:
    df[col] = df[col].str.strip().str.strip('"')

print('Categorias apos padronizacao:')
for col in ['Vehicle Type', 'periodo_dia']:
    print(f'\n{col} ({df[col].nunique()} valores):')
    print(df[col].value_counts().to_string())

print(f'\nPickup Location: {df["Pickup Location"].nunique()} valores unicos')
print(f'Drop Location:   {df["Drop Location"].nunique()} valores unicos')

## 4. Validação final e salvar dataset tratado

Conferir shape, tipos, nulos e distribuição do alvo antes de persistir.

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# Resumo final
print('=== RESUMO DO DATASET TRATADO ===')
print(f'Linhas:   {len(df):,}')
print(f'Colunas:  {len(df.columns)}')
print()
print('Tipos de dados:')
print(df.dtypes)
print()
print('Nulos por coluna:')
print(df.isna().sum())
print()
print('Distribuicao do alvo:')
vc = df['alvo'].value_counts().sort_index()
print(f'  Nao concluida (0): {vc[0]:,}  ({vc[0]/len(df):.1%})')
print(f'  Concluida     (1): {vc[1]:,}  ({vc[1]/len(df):.1%})')

# Salvar
df.to_csv('data/dataset_tratado.csv', index=False)
print()
print('Salvo em: data/dataset_tratado.csv')
df.head()